# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vincentoei/flyrank-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Note on the output:** the warehouse label base rate (~24.7%) is higher than the starter proxy (~7.8%) from `w01`/`w02`. This is expected because the warehouse filters to higher-volume pages (`gsc_impressions_90d >= 300`) and uses a concrete future month (March → April 2026), while the starter proxy was a snapshot approximation. Both share the same 20%-growth threshold, but the underlying populations differ.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import getpass
import duckdb
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score

# Load token
token = os.getenv("HF_TOKEN")
if not token:
    token = getpass.getpass("Hugging Face token: ")

conn = duckdb.connect()
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")
conn.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE HTTP,
        EXTRA_HTTP_HEADERS MAP {{
            'Authorization': 'Bearer {token}'
        }}
    );
""")

conn.execute("""
    CREATE OR REPLACE VIEW fact AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
""")

conn.execute("""
    CREATE OR REPLACE VIEW dim_content AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
""")

conn.execute("""
    CREATE OR REPLACE VIEW dim_clients AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
""")

print("Warehouse views attached.")

feature_vector = conn.execute("""
    WITH feature_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS gsc_impressions_90d,
            AVG(gsc_avg_position) AS gsc_avg_position_90d,
            SUM(gsc_clicks) AS gsc_clicks_90d,
            CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) ELSE 0 END AS gsc_ctr_90d,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions_organic ELSE 0 END) AS sessions_organic_90d,
            COUNT(DISTINCT report_date) AS days_with_impressions_90d
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-0[1-3]/*.parquet')
        WHERE report_date BETWEEN '2026-01-01' AND '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    ),
    last_30 AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS gsc_impressions_last_30d
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    ),
    label_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS gsc_impressions_next_30d
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')
        WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
        GROUP BY client_hash_id, content_hash_id
    ),
    decision_date AS (
        SELECT DATE '2026-03-31' AS d
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.gsc_impressions_90d,
        f.gsc_avg_position_90d,
        f.gsc_clicks_90d,
        f.gsc_ctr_90d,
        f.sessions_organic_90d,
        f.days_with_impressions_90d,
        l30.gsc_impressions_last_30d,
        l.gsc_impressions_next_30d,
        dim_content.search_volume,
        dim_content.competition,
        dim_content.cpc,
        dim_content.word_count,
        dim_content.content_type,
        dim_content.main_intent,
        (d.d - dim_content.content_created_date)::INTEGER AS content_age_days,
        CASE
            WHEN l.gsc_impressions_next_30d >= 1.20 * l30.gsc_impressions_last_30d
                 AND l30.gsc_impressions_last_30d >= 100
            THEN 1 ELSE 0
        END AS growth_label
    FROM feature_window f
    LEFT JOIN last_30 l30
        ON f.content_hash_id = l30.content_hash_id
    LEFT JOIN label_window l
        ON f.content_hash_id = l.content_hash_id
    LEFT JOIN dim_content
        ON f.content_hash_id = dim_content.content_hash_id
    CROSS JOIN decision_date d
    WHERE f.gsc_impressions_90d >= 300
      AND dim_content.content_created_date <= d.d
      AND dim_content.word_count IS NOT NULL
""").df()

print(f"Feature vector rows: {len(feature_vector):,}")
print(f"Positive growth rate: {feature_vector['growth_label'].mean():.3%}")
print(feature_vector.head())

# --- Sanity checks on the feature vector ---
print("\n" + "=" * 60)
print("Sanity checks:")
print(f"Rows: {len(feature_vector):,}")
print(f"Positive growth rate: {feature_vector['growth_label'].mean():.3%}")
print(f"\nNumeric feature summary:")
print(feature_vector[['gsc_impressions_90d', 'gsc_impressions_last_30d', 'gsc_impressions_next_30d',
                       'gsc_avg_position_90d', 'word_count', 'content_age_days']].describe().round(2))


Warehouse views attached.
Feature vector rows: 67,478
Positive growth rate: 24.731%
            client_hash_id           content_hash_id  gsc_impressions_90d  \
0  client_62f4a7e64f5e0096  content_66e069c4eea5f9eb                566.0   
1  client_62f4a7e64f5e0096  content_cdf63a3a7f20a480               3002.0   
2  client_62f4a7e64f5e0096  content_f6456bfee1a23500               7609.0   
3  client_62f4a7e64f5e0096  content_8927cb17001cbe76               2888.0   
4  client_62f4a7e64f5e0096  content_cd74b99dc5d1a25a              13200.0   

   gsc_avg_position_90d  gsc_clicks_90d  gsc_ctr_90d  sessions_organic_90d  \
0              3.831741             0.0     0.000000                   0.0   
1              5.861784             0.0     0.000000                   0.0   
2              4.869760            13.0     0.001709                   0.0   
3              3.916303             7.0     0.002424                   0.0   
4              3.587635            76.0     0.005758           

## 2. Feature notes (meaning, missing, categorical, available-when?)

Each feature below is knowable strictly before the decision date 2026-03-31. The target variable is the only column that looks forward.

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| `gsc_impressions_90d` | Total Google Search impressions, 2026-01-01 to 2026-03-31 | 0 if no rows; we keep only pages with >= 300 impressions to reduce noise | Yes, before decision |
| `gsc_avg_position_90d` | Average GSC position over the 90-day feature window | Averaged over existing rows; lower is better | Yes |
| `gsc_clicks_90d` | GSC clicks over the 90-day window | 0 if no clicks | Yes |
| `gsc_ctr_90d` | `gsc_clicks_90d / gsc_impressions_90d * 100` | 0 when impressions are 0 | Yes |
| `sessions_organic_90d` | Organic GA4 sessions over the 90-day window | Summed only when `ga4_data_available IS TRUE`; otherwise 0 | Yes, with caveat — GA4 coverage is sparse |
| `days_with_impressions_90d` | Number of days with at least one impression | 0 if no data | Yes |
| `gsc_impressions_last_30d` | GSC impressions in March 2026 only | 0 if no March data | Yes, available exactly on decision date |
| `search_volume` | Estimated search volume for the target keyword | Static content metadata from `dim_content` | Yes |
| `competition` | Keyword competition score, 0–1 | Static content metadata | Yes |
| `cpc` | Cost-per-click estimate | Static content metadata | Yes |
| `word_count` | Article word count | Static content metadata; rows with null word_count are dropped | Yes |
| `content_type` | Page type (e.g. keyword article, comparison article) | Categorical from `dim_content` | Yes |
| `main_intent` | Search intent (informational, transactional, etc.) | Categorical from `dim_content` | Yes |
| `content_age_days` | Days since `content_created_date` | Derived from `dim_content` at the decision date | Yes |
| `gsc_impressions_next_30d` | GSC impressions in April 2026 (the exact window being predicted) | 0 if no April data | **Future** — never a feature |
| `growth_label` | **Target / proxy** | 1 if April 2026 impressions are >= 1.2x March 2026 and March has >= 100 impressions | **Future** — never a feature |


In [2]:
# Missing-value audit and categorical summary for the feature vector
print("Missing values per column:")
print(feature_vector.isnull().mean().round(3).to_string())

print("\n" + "=" * 60)
print("Categorical value counts:")
for col in ["content_type", "main_intent"]:
    if col in feature_vector.columns:
        print(f"\n{col}:")
        print(feature_vector[col].value_counts(dropna=False).head(10).to_string())


Missing values per column:
client_hash_id               0.000
content_hash_id              0.000
gsc_impressions_90d          0.000
gsc_avg_position_90d         0.000
gsc_clicks_90d               0.000
gsc_ctr_90d                  0.000
sessions_organic_90d         0.000
days_with_impressions_90d    0.000
gsc_impressions_last_30d     0.045
gsc_impressions_next_30d     0.045
search_volume                0.021
competition                  0.021
cpc                          0.021
word_count                   0.000
content_type                 0.000
main_intent                  0.020
content_age_days             0.000
growth_label                 0.000

Categorical value counts:

content_type:
content_type
keyword article       66158
feedly article          870
comparison article      450

main_intent:
main_intent
informational    40051
transactional    13863
commercial       12018
NaN               1370
navigational       176


## 3. The leakage hunt

Three attacks on the feature vector:

1. **Label-derived feature attack.** Add `gsc_impressions_next_30d` (the exact April 2026 impressions we are trying to predict) as a feature. If the score jumps sharply, the test harness is sensitive to leakage and we know what to remove.
2. **Overlapping-window check.** Confirm that `gsc_impressions_90d` and `gsc_impressions_last_30d` sum only pre-decision dates.
3. **Product-flag / decision-derived check.** Confirm that no `health_score`, `priority_score`, or `action_type` columns are in the feature list — those are not even in the release, but the habit matters.

The honest model is trained on the 12 numeric features. The leaky model is trained on the same features plus `gsc_impressions_next_30d`. The gap between the two AUCs is the leakage confession.


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

# Numeric features available strictly before the decision date
features = [
    "gsc_impressions_90d",
    "gsc_avg_position_90d",
    "gsc_clicks_90d",
    "gsc_ctr_90d",
    "sessions_organic_90d",
    "days_with_impressions_90d",
    "gsc_impressions_last_30d",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "content_age_days",
]

# --- Honest model ---
X = feature_vector[features].replace([float("inf"), float("-inf")], 0).fillna(0)
y = feature_vector["growth_label"]

honest_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
honest_model.fit(X, y)
honest_auc = roc_auc_score(y, honest_model.predict_proba(X)[:, 1])

print(f"Honest AUC (12 features): {honest_auc:.3f}")

# --- Leaky model: add the future impressions ---
leaky_features = features + ["gsc_impressions_next_30d"]
X_leaky = feature_vector[leaky_features].replace([float("inf"), float("-inf")], 0).fillna(0)

leaky_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
leaky_model.fit(X_leaky, y)
leaky_auc = roc_auc_score(y, leaky_model.predict_proba(X_leaky)[:, 1])

print(f"Leaky AUC (with future impressions): {leaky_auc:.3f}")
print(f"AUC lift from leakage: {leaky_auc - honest_auc:+.3f}")

# --- Feature importance sanity check ---
fi = pd.DataFrame({
    "feature": features,
    "importance": honest_model.feature_importances_,
}).sort_values("importance", ascending=False)

print("\nHonest feature importances:")
print(fi.to_string(index=False))

# --- Overlapping-window check: verify the 90d window never touches April ---
window_check = conn.execute("""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-0[1-3]/*.parquet')
    WHERE report_date BETWEEN '2026-01-01' AND '2026-03-31'
""").df()

print("\nFeature window date span:")
print(window_check.to_string(index=False))
assert window_check["max_date"].iloc[0] <= pd.Timestamp("2026-03-31"), "Feature window leaks into April!"


Honest AUC (12 features): 0.786
Leaky AUC (with future impressions): 0.938
AUC lift from leakage: +0.152

Honest feature importances:
                  feature  importance
         content_age_days    0.284587
days_with_impressions_90d    0.231178
              gsc_ctr_90d    0.162959
               word_count    0.093027
 gsc_impressions_last_30d    0.060377
      gsc_impressions_90d    0.042288
           gsc_clicks_90d    0.036976
     sessions_organic_90d    0.035365
     gsc_avg_position_90d    0.023071
            search_volume    0.016076
                      cpc    0.008033
              competition    0.006063

Feature window date span:
  min_date   max_date
2026-01-01 2026-03-31


## 4. What I excluded and why

These fields are either unavailable at the decision date, derived from the target, or product-decision outputs that would create circular reasoning.

| Field / group | Why excluded |
|---|---|
| `gsc_impressions_next_30d` | Target-window metric — it is the exact future outcome we are predicting. |
| `trend_direction`, `trend_pct` | Derived from the same label window in the starter CSV; using them leaks the answer. |
| `content_updated_date`, `last_optimized_date` | Warehouse refresh/system timestamps, often after 2026-03-31; not knowable at the decision moment. |
| `fact_content_query_90d` columns | The fixed 90-day query window overlaps April 2026, so it would contain target-window information. |
| `health_score`, `priority_score`, `action_type` | Product decision flags. They are not present in the release, but even if rebuilt they would be circular to use as features. |
| `client_hash_id`, `content_hash_id` | Context only — used for joining and grouping, never as model inputs. |

**Data quality limits also noted:** some pages have `content_age_days < 90`, meaning they were created mid-window. Their `gsc_impressions_90d` is real but accumulated over fewer than 90 days. This is a limitation, not leakage, and should be mentioned in the report.

In [4]:
# Data quality check: pages created mid-window
print(f"Pages with content_age_days < 90: {(feature_vector['content_age_days'] < 90).sum():,} ({(feature_vector['content_age_days'] < 90).mean():.1%})")
print(f"Min content_age_days: {feature_vector['content_age_days'].min()}")
print()

# Confirm no excluded columns are in the feature list
excluded_from_features = {
    "gsc_impressions_next_30d",
    "trend_direction",
    "trend_pct",
    "content_updated_date",
    "last_optimized_date",
    "health_score",
    "priority_score",
    "action_type",
}

used = set(features)
violations = used & excluded_from_features

print(f"Features used: {len(used)}")
print(f"Excluded fields: {len(excluded_from_features)}")
print(f"Violations: {len(violations)}")

assert len(violations) == 0, f"Leakage risk: excluded fields found in feature list -> {violations}"

# Context IDs and the target column live in the DataFrame but are not in the model feature list.
print(f"feature_vector context columns: client_hash_id, content_hash_id")
print(f"feature_vector target column: growth_label")
print("Neither context nor target is in the model feature list — confirmed.")


Features used: 12
Excluded fields: 8
Violations: 0
feature_vector context columns: client_hash_id, content_hash_id
feature_vector target column: growth_label
Neither context nor target is in the model feature list — confirmed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.